In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_recall_curve, auc, precision_score, recall_score, confusion_matrix

from sklearn.experimental import enable_iterative_imputer 
from sklearn.impute import IterativeImputer

# -----------------------------
# Define target and input
# -----------------------------
X = df_clean.drop('boycott', axis=1)
y = df_clean['boycott'].astype(int)

# -----------------------------
# Nested CV with metrics + confusion matrices
# -----------------------------
def nested_cv_with_metrics(X, y, use_class_weight=False, plot_avg_cm=True):
    random_state = 42
    outer_splits = 5
    inner_splits = 5

    # Detect column types
    numerical_cols = X.select_dtypes(include=['float64']).columns
    categorical_cols = X.select_dtypes(include=['object']).columns

    # Preprocessor
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('imputer', IterativeImputer(random_state=42, max_iter=10, sample_posterior=True)),
                ('scaler', StandardScaler())
            ]), numerical_cols),
            ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
        ]
    )

    # -----------------------------
    # MODELS
    # -----------------------------
    models = {
        "Logistic Regression": {
            "model": LogisticRegression(
                max_iter=2000,
                solver='liblinear',
                random_state=random_state,
                class_weight='balanced' if use_class_weight else None
            ),
            "params": {"model__C": [0.1, 1.0, 10.0]}
        },
        "Random Forest": {
            "model": RandomForestClassifier(
                n_estimators=200,
                random_state=random_state,
                class_weight='balanced' if use_class_weight else None
            ),
            "params": {
                "model__max_depth": [5, 10, None],
                "model__min_samples_split": [2, 5, 10]
            }
        }
    }

    outer_cv = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=random_state)
    inner_cv = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=random_state)

    results = {}
    pr_curves = {}
    conf_matrices = {}  # NEW: store confusion matrices

    for name, spec in models.items():

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", spec["model"])
        ])

        # Storage
        outer_f1 = []
        outer_pr_auc = []
        outer_precision = []
        outer_recall = []
        inner_f1_all = []
        inner_pr_auc_all = []
        outer_precisions = []
        outer_recalls = []

        # NEW: store per-fold confusion matrices
        conf_matrices[name] = []

        for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), 1):

            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # -----------------------------
            # Inner GridSearch
            # -----------------------------
            grid = GridSearchCV(
                pipeline,
                param_grid=spec["params"],
                cv=inner_cv,
                scoring='f1'
            )
            grid.fit(X_train, y_train)
            best_model = grid.best_estimator_
            inner_f1_all.append(grid.best_score_)

            # -----------------------------
            # Inner PR-AUC
            # -----------------------------
            inner_pr_fold = []
            for inner_train_idx, inner_val_idx in inner_cv.split(X_train, y_train):
                X_inner_train = X_train.iloc[inner_train_idx]
                X_inner_val = X_train.iloc[inner_val_idx]
                y_inner_train = y_train.iloc[inner_train_idx]
                y_inner_val = y_train.iloc[inner_val_idx]

                best_model.fit(X_inner_train, y_inner_train)
                y_prob_inner = best_model.predict_proba(X_inner_val)[:,1]
                precision_inner, recall_inner, _ = precision_recall_curve(y_inner_val, y_prob_inner)
                inner_pr_auc_fold = auc(recall_inner, precision_inner)
                inner_pr_fold.append(inner_pr_auc_fold)

            inner_pr_auc_all.append(np.mean(inner_pr_fold))

            # -----------------------------
            # Outer predictions
            # -----------------------------
            y_pred = best_model.predict(X_test)
            y_prob = best_model.predict_proba(X_test)[:,1]

            outer_f1.append(f1_score(y_test, y_pred))
            outer_precision.append(precision_score(y_test, y_pred))
            outer_recall.append(recall_score(y_test, y_pred))

            precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_prob)
            pr_auc = auc(recall_vals, precision_vals)
            outer_pr_auc.append(pr_auc)
            outer_precisions.append(precision_vals)
            outer_recalls.append(recall_vals)

            # -----------------------------
            # NEW: store confusion matrix per fold
            # -----------------------------
            cm = confusion_matrix(y_test, y_pred)
            conf_matrices[name].append(cm)

        # -----------------------------
        # Store results
        # -----------------------------
        results[name] = {
            "inner_f1": inner_f1_all,
            "inner_pr_auc": inner_pr_auc_all,
            "outer_f1": outer_f1,
            "outer_pr_auc": outer_pr_auc,
            "outer_precision": outer_precision,
            "outer_recall": outer_recall
        }
        pr_curves[name] = (outer_precisions, outer_recalls)

        # -----------------------------
        # Print summary
        # -----------------------------
        print(f"\n{name} {'with' if use_class_weight else 'without'} class weighting")
        print(f"Inner F1: {np.mean(inner_f1_all):.3f} ± {np.std(inner_f1_all):.3f}")
        print(f"Inner PR-AUC: {np.mean(inner_pr_auc_all):.3f} ± {np.std(inner_pr_auc_all):.3f}")
        print(f"Outer F1: {np.mean(outer_f1):.3f} ± {np.std(outer_f1):.3f}")
        print(f"Outer PR-AUC: {np.mean(outer_pr_auc):.3f} ± {np.std(outer_pr_auc):.3f}")
        print(f"Outer Precision: {np.mean(outer_precision):.3f} ± {np.std(outer_precision):.3f}")
        print(f"Outer Recall: {np.mean(outer_recall):.3f} ± {np.std(outer_recall):.3f}")

    # -----------------------------
    # Plot PR curves
    # -----------------------------
    for name, (precs, recs) in pr_curves.items():
        plt.figure(figsize=(7,6))
        for i in range(len(precs)):
            plt.plot(recs[i], precs[i], alpha=0.4, label=f'Fold {i+1} (AUC={results[name]["outer_pr_auc"][i]:.2f})')

        mean_recall = np.linspace(0,1,100)
        interp_precs = [np.interp(mean_recall, r[::-1], p[::-1]) for p,r in zip(precs,recs)]
        mean_precision = np.mean(interp_precs, axis=0)
        plt.plot(mean_recall, mean_precision, color='black', linewidth=2, label='Mean PR Curve')

        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title(f"Precision-Recall Curve - {name} {'with' if use_class_weight else 'without'} weighting")
        plt.legend()
        plt.grid(True)
        plt.show()

    # -----------------------------
    # BAR CHART: Performance Metrics
    # -----------------------------
    model_names = list(results.keys())
    metrics = {
        "F1-score": "outer_f1",
        "PR-AUC": "outer_pr_auc",
        "Precision": "outer_precision",
        "Recall": "outer_recall"
    }

    means = {metric: [np.mean(results[m][key]) for m in model_names] for metric, key in metrics.items()}
    stds = {metric: [np.std(results[m][key]) for m in model_names] for metric, key in metrics.items()}
    colors = plt.cm.viridis(np.linspace(0.3,0.8,len(model_names)))

    fig, axes = plt.subplots(2,2, figsize=(12,8))
    axes = axes.flatten()

    for ax, (metric_name, _) in zip(axes, metrics.items()):
        bars = ax.bar(model_names, means[metric_name], yerr=stds[metric_name], capsize=6, color=colors, alpha=0.9)
        ax.set_title(metric_name, fontsize=13, fontweight='bold')
        ax.grid(axis='y', linestyle='--', alpha=0.6)

        for bar, mean in zip(bars, means[metric_name]):
            ax.text(bar.get_x()+bar.get_width()/2, mean-0.05, f"{mean:.3f}", ha='center', fontsize=11, fontweight='bold')

        for spine in ax.spines.values():
            spine.set_visible(False)

    fig.suptitle(f"Model Performance {'with' if use_class_weight else 'without'} Class Weighting", fontsize=15, fontweight='bold')
    plt.tight_layout(rect=[0,0,1,0.95])
    plt.show()

    # -----------------------------
    # Return results + confusion matrices
    # -----------------------------
    return results, conf_matrices

# -----------------------------
# RUN
# -----------------------------
# Without weighting
results_no_weight, conf_no_weight = nested_cv_with_metrics(X, y, use_class_weight=False)

# With weighting
results_weight, conf_weight = nested_cv_with_metrics(X, y, use_class_weight=True)

# -----------------------------
# OPTIONAL: Plot Average Normalized Confusion Matrices
# -----------------------------
def plot_avg_normalized_cm(conf_matrices_dict):
    for key, cms in conf_matrices_dict.items():
        n_folds = len(cms)
        cms_normalized = np.array([cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] for cm in cms])
        cm_avg = np.mean(cms_normalized, axis=0)
        plt.figure(figsize=(5,4))
        sns.heatmap(cm_avg, annot=True, fmt='.2f', cmap='Blues', cbar=False)
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.title(f'Average Normalized Confusion Matrix\n{key}')
        plt.show()

# Combine the weighting/no-weighting runs
combined_conf_matrices = {}
for model in conf_no_weight.keys():
    combined_conf_matrices[f"{model} - No Weight"] = conf_no_weight[model]
    combined_conf_matrices[f"{model} - With Weight"] = conf_weight[model]

# Plot 4 average normalized confusion matrices
plot_avg_normalized_cm(combined_conf_matrices)